In [ ]:
#this notebook is EDA to look at whether or not predicting time to produce video makes sense

In [ ]:
#%pip install duckdb


In [3]:
import duckdb
import numpy 
import pandas as pd


In [4]:
con = duckdb.connect('../data/workload.duckdb')
videos = con.execute("""
    SELECT
        *
    FROM dim_videos
""").df()
con.close()


In [5]:
pd.set_option('display.max_rows', 200)

In [19]:
pd.set_option('display.max_columns', 200)

In [19]:
videos[['media_type','media_title', 'video_type', 'video_subtype','total_hours', 'total_day_span', 'active_days_worked']].head(200)

,media_type,media_title,video_type,video_subtype,total_hours,total_day_span,active_days_worked
0,Video Games,Elden Ring,Playthrough,Part 5,24.20,15,10
1,Video Games,Elden Ring,Review,Demi Human Chief,8.02,2,2
2,Video Games,Elden Ring,Review,Erdtree Burial Watchdog,8.42,13,4
3,Video Games,Elden Ring,Playthrough,Part 6,18.48,19,9
4,Video Games,Elden Ring,Review,Mad Pumpkin Head,5.35,2,2
...,...,...,...,...,...,...,...
120,Movies,A Nightmare on Elm Street,Stats,Nightmare on Elm Street,0.62,4,2
121,Movies,A Nightmare on Elm Street,Review,3 Things,24.47,35,20
122,Movies,A Nightmare on Elm Street 2: Freddy's Revenge,Review,3 Things,3.02,11,5
123,Movies,A Nightmare on Elm Street,Scene Breakdown,Freddy's Introduction,8.70,4,4


In [ ]:
#initial dbt model dim_movies has script as part of processing.  I have switched it to pre-processing
# because i think there could be a useful signal in terms of which videos i begin editing 
#while the script is still in flux

#overall some clear signals above, big difference sbetween movies andd video games.
#but going forward i will need to delete the uncompleted videos.

#and the total day span seems to not make sense.  i have to figure out how to calculate overlapping
#videos.  


In [ ]:
videos.info()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 35 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   video_id                        125 non-null    str           
 1   media_title                     125 non-null    str           
 2   video_type                      125 non-null    str           
 3   video_subtype                   125 non-null    str           
 4   media_type                      125 non-null    str           
 5   media_series                    125 non-null    str           
 6   total_hours                     125 non-null    float64       
 7   date_first                      125 non-null    datetime64[us]
 8   date_last                       125 non-null    datetime64[us]
 9   total_day_span                  125 non-null    int64         
 10  active_days_worked              125 non-null    int64         
 11  hours_editing    

In [20]:
videos[['media_type','media_title', 'video_type', 'video_subtype','total_hours', 'hours_pre_processing', 'hours_processing','hours_post_processing']].head(20)

,media_type,media_title,video_type,video_subtype,total_hours,hours_pre_processing,hours_processing,hours_post_processing
0,Video Games,Elden Ring,Playthrough,Part 5,24.20,2.80,20.28,1.12
1,Video Games,Elden Ring,Review,Demi Human Chief,8.02,0.82,6.68,0.52
2,Video Games,Elden Ring,Review,Erdtree Burial Watchdog,8.42,0.48,6.82,1.12
3,Video Games,Elden Ring,Playthrough,Part 6,18.48,4.22,14.05,0.22
4,Video Games,Elden Ring,Review,Mad Pumpkin Head,5.35,0.40,4.32,0.63
5,Video Games,Elden Ring,Playthrough,Part 7,0.25,0.25,0.00,0.00
6,Video Games,Elden Ring,Review,Black Knife Assassin,5.83,0.65,4.78,0.40
7,Video Games,Elden Ring,Review,Beastman of Farum Azula,8.68,0.88,7.25,0.55
8,Video Games,The Witcher 3,Review,Twisted Firestarter,10.23,1.08,7.98,1.17
9,Video Games,The Witcher 3,Review,Kaer Morhen,18.70,5.47,12.12,1.12


In [ ]:
###Remove incomplete videos

media_title	video_type	video_subtype
Elden Ring	Playthrough	Part 7
Elden Ring	Review	Godrick the Grafted
Elden Ring	Playthrough	Intro
Elden Ring	Playthrough	Stormhill
Elden Ring	Playthrough	Limgrave Dungeons
Elden Ring	Playthrough	Lake Agheel
Elden Ring	Playthrough	Fringefolk
Grand Theft Auto V	Review	Trevor's Intro
Elden Ring	Playthrough	Deaths 03
	        Shorts	
	        Essay	
Terminator Movies		
Man of Steel		
Halloween	Scene Breakdown	
Hitman WOA	Playthrough	Sapienza Master
Hitman WOA	Playthrough	Paris Silenced Pistol


In [12]:

remove_mask = (
    # Specific (title, type, subtype) combinations
    videos[['media_title', 'video_type', 'video_subtype']].apply(tuple, axis=1).isin({
        ('Elden Ring', 'Playthrough', 'Part 7'),
        ('Elden Ring', 'Review', 'Godrick the Grafted'),
        ('Elden Ring', 'Playthrough', 'Intro'),
        ('Elden Ring', 'Playthrough', 'Stormhill'),
        ('Elden Ring', 'Playthrough', 'Limgrave Dungeons'),
        ('Elden Ring', 'Playthrough', 'Lake Agheel'),
        ('Elden Ring', 'Playthrough', 'Fringefolk'),
        ('Grand Theft Auto V', 'Review', "Trevors Intro"),
        ('Elden Ring', 'Playthrough', 'Deaths 03'),
        ('Hitman WOA', 'Playthrough', 'Sapienza Master'),
        ('Hitman WOA', 'Playthrough', 'Paris Silenced Pistol'),
        ('Grand Theft Auto V', 'Review', "Trevor's Intro"),
        ('The Witcher 3', 'Rankings', 'Velen Main Quests')
    })
    # Any row typed as Shorts or Essay (no complete videos in these categories)
    | videos['video_type'].isin(['Shorts', 'Essay', 'Stats'])
    # Entire titles that are incomplete
    | videos['media_title'].isin(['Terminator Movies', 'Man of Steel',"Marvel's Spiderman"])
    # Halloween scene breakdown
    | ((videos['media_title'] == 'Halloween') & (videos['video_type'] == 'Scene Breakdown'))
)

print("Rows being removed:")
print(videos[remove_mask][['media_title', 'video_type', 'video_subtype']].to_string())
print(f"\nOriginal: {len(videos)} | Removed: {remove_mask.sum()} | Remaining: {len(videos) - remove_mask.sum()}")
videos_complete = videos[~remove_mask].reset_index(drop=True)


Rows being removed:
                                 media_title       video_type                         video_subtype
5                                 Elden Ring      Playthrough                                Part 7
13                        Marvel's Spiderman           Review                        The Main Event
14                                Elden Ring           Review                   Godrick the Grafted
42                                Elden Ring      Playthrough                                 Intro
43                                Elden Ring      Playthrough                             Stormhill
44                                Elden Ring      Playthrough                     Limgrave Dungeons
47                                Elden Ring      Playthrough                           Lake Agheel
48                                Elden Ring      Playthrough                            Fringefolk
57                        Grand Theft Auto V           Review                   

In [34]:
videos_complete[['media_title', 'video_type', 'video_subtype', 'total_hours', 'hours_pre_processing', 'hours_processing', 'hours_post_processing']]

,media_title,video_type,video_subtype,total_hours,hours_pre_processing,hours_processing,hours_post_processing
0,Elden Ring,Playthrough,Part 5,24.20,2.80,20.28,1.12
1,Elden Ring,Review,Demi Human Chief,8.02,0.82,6.68,0.52
2,Elden Ring,Review,Erdtree Burial Watchdog,8.42,0.48,6.82,1.12
3,Elden Ring,Playthrough,Part 6,18.48,4.22,14.05,0.22
4,Elden Ring,Review,Mad Pumpkin Head,5.35,0.40,4.32,0.63
...,...,...,...,...,...,...,...
99,Indiana Jones and the Kingdom of the Crystal S...,Scene Breakdown,Prologue,13.58,1.12,11.85,0.62
100,A Nightmare on Elm Street,Review,3 Things,24.47,3.35,19.83,1.28
101,A Nightmare on Elm Street 2: Freddy's Revenge,Review,3 Things,3.02,2.47,0.55,0.00
102,A Nightmare on Elm Street,Scene Breakdown,Freddy's Introduction,8.70,0.37,6.87,1.47


In [17]:
videos.head()

,video_id,media_title,video_type,video_subtype,media_type,media_series,total_hours,date_first,date_last,total_day_span,active_days_worked,hours_editing,hours_processing_raw_video,hours_recording_audio,hours_recording_video,hours_script,hours_subtitles,hours_thumbnail,hours_uploading,hours_pre_processing,hours_processing,hours_post_processing,hours_uncategorised,pre_processing_date_first,pre_processing_date_last,pre_processing_total_day_span,pre_processing_active_days,processing_date_first,processing_date_last,processing_total_day_span,processing_active_days,post_processing_date_first,post_processing_date_last,post_processing_total_day_span,post_processing_active_days
0,3f1e15377df0aa1889f360c247df7a90,Elden Ring,Playthrough,Part 5,Video Games,Soulsborne Games,24.20,2023-02-13,2023-02-27,15,10,14.80,0.05,5.48,0.00,2.75,0.0,0.50,0.62,2.80,20.28,1.12,0.0,2023-02-13,2023-02-23,11,4,2023-02-13,2023-02-25,13,9,2023-02-24,2023-02-27,4,3
1,8b355e78a5e31e75061372e25b22d6c7,Elden Ring,Review,Demi Human Chief,Video Games,Soulsborne Games,8.02,2023-02-15,2023-02-16,2,2,5.80,0.37,0.88,0.00,0.45,0.0,0.00,0.52,0.82,6.68,0.52,0.0,2023-02-15,2023-02-15,1,1,2023-02-15,2023-02-16,2,2,2023-02-16,2023-02-16,1,1
2,da6e01691e06c291bc49a6dc16d9315e,Elden Ring,Review,Erdtree Burial Watchdog,Video Games,Soulsborne Games,8.42,2023-02-18,2023-03-02,13,4,4.32,0.05,2.50,0.00,0.43,0.0,0.82,0.30,0.48,6.82,1.12,0.0,2023-02-18,2023-02-28,11,2,2023-02-28,2023-03-02,3,3,2023-02-28,2023-03-02,3,2
3,e63306ce56de41f88ed7de3e3623aad9,Elden Ring,Playthrough,Part 6,Video Games,Soulsborne Games,18.48,2023-03-06,2023-03-24,19,9,8.90,0.37,4.63,0.52,3.85,0.0,0.22,0.00,4.22,14.05,0.22,0.0,2023-03-06,2023-03-15,10,7,2023-03-07,2023-03-24,18,8,2023-03-07,2023-03-07,1,1
4,1050d995453283fca97699b97994f646,Elden Ring,Review,Mad Pumpkin Head,Video Games,Soulsborne Games,5.35,2023-03-08,2023-03-09,2,2,3.05,0.00,1.27,0.00,0.40,0.0,0.37,0.27,0.40,4.32,0.63,0.0,2023-03-08,2023-03-08,1,1,2023-03-08,2023-03-09,2,2,2023-03-08,2023-03-09,2,2


In [26]:
videos_complete['media_type'].value_counts()

media_type
Video Games    58
Movies         47
Name: count, dtype: int64

In [27]:
videos_complete['media_series'].value_counts()

media_series
The Witcher Series                  38
Indiana Jones Series                14
Soulsborne Games                    12
Jurassic Park Series                11
Halloween Series                     7
Superman Series                      6
Jaws Series                          5
GTA Series                           4
A Nightmare on Elm Street Series     4
Baldur's Gate Series                 2
Hitman Series                        2
Name: count, dtype: int64

In [ ]:
#media series is too sparse.  I think the way to capture this is either flag when its the first in a series or a # of months since the last video in the series



In [28]:
videos_complete['video_type'].value_counts()

video_type
Review             74
Rankings           16
Playthrough         8
Scene Breakdown     5
Retrospectives      1
Stats               1
Name: count, dtype: int64

In [ ]:
#for the above "Scene Breakdown" is specific to movies, "Playthrough" is specific to video games.  The primary signals will be "Review" and "Rankings".
# so for now probably can group the other 4 into "Other"



In [50]:
videos_complete.groupby('video_type')[['hours_pre_processing', 'hours_processing']].mean()


,hours_pre_processing,hours_processing
video_type,,
Playthrough,3.39500,13.881250
Rankings,1.00500,11.172500
Retrospectives,0.85000,11.250000
Review,3.44973,14.081757
Scene Breakdown,1.04000,9.474000


In [35]:
videos_complete['video_type']=videos_complete['video_type'].replace({"Playthrough":"Other",
                                       'Scene Breakdown':"Other",
                                       'Retrospectives':"Other",
                                       'Stats':"Other"})

In [36]:
videos_complete['video_type'].value_counts()

video_type
Review      74
Rankings    16
Other       14
Name: count, dtype: int64

In [38]:
videos_complete['video_subtype'].value_counts().head(20)

video_subtype
3 Things                    26
Scene Rankings               9
Prologue                     5
Movie Series By Category     3
Part 5                       1
Demi Human Chief             1
Erdtree Burial Watchdog      1
Part 6                       1
Mad Pumpkin Head             1
Black Knife Assassin         1
Beastman of Farum Azula      1
Twisted Firestarter          1
Kaer Morhen                  1
Tree Sentinel                1
Ulcerated Tree Spirit        1
Margit the Fell Omen         1
Precious Cargo               1
Devil By the Well            1
Lilac and Gooseberries       1
Funeral Pyres                1
Name: count, dtype: int64

In [ ]:
#i think video subtype is too granular, particularly for the video games.  i think the main signal will be in "video_type"

In [43]:
pd.crosstab(videos_complete['media_type'], videos_complete['video_type'], normalize='index')

video_type,Other,Rankings,Review
media_type,,,
Movies,0.130435,0.282609,0.586957
Video Games,0.137931,0.051724,0.810345


In [44]:
pd.crosstab(videos_complete['media_type'], videos_complete['video_type'])

video_type,Other,Rankings,Review
media_type,,,
Movies,6,13,27
Video Games,8,3,47


In [ ]:
#These are the variables  "video_type" and "media_type".  We do have plenty that is below 30 (or even 20 or 10)
#but im not sure if i should break it down by "media_type"
#i think the more useful signals will be if i apply some metadata to these videos.  
# "expected_length (Long, Short)",
#  "expected_complexity (High, Low)", 
# "first_of_type".  This one is not LITERALLY first of type but is more an indication that there are "start_up" problems  and can represent
# it being the fisrt of a specific type of video, first time i tried something, etc
# 
#   


In [10]:
#create a list of videos for a csv seed
videos[['video_id', 'media_title','video_type','video_subtype']].drop_duplicates().to_clipboard(index=False)

In [ ]:
#Next steps
#EDA on pre-processing and processing time and combined
#Generate "Expected Length" and "Expected Complexity" variables.  Develop a rubrick for the expected complexity.